In [19]:
# =============================================================
# DARKLINE — NOTEBOOK B : ENGINES AND EXPORT
# Team CORTEX · Accenture Innovation Challenge 2026
# =============================================================
# Consumes Notebook A's output. Produces the JSON bundle that the
# dashboard reads. Nothing here is slow; every cell is minutes.
# =============================================================

import os, re, gc, json, time, math, zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

BASE = Path('/kaggle/input/bosch-production-line-performance')
if not BASE.exists():
    BASE = Path('/kaggle/input/competitions/bosch-production-line-performance')
NUM, DAT, CAT = BASE/'train_numeric.csv.zip', BASE/'train_date.csv.zip', BASE/'train_categorical.csv.zip'

# ---- find Notebook A's output ------------------------------------
A_OUT = None
for p in Path('/kaggle/input').rglob('station_times.parquet'):
    A_OUT = p.parent
    break
assert A_OUT is not None, (
    "Notebook A's output not attached. Do: Add Data -> Your Work -> "
    "select the Notebook A output that contains station_times.parquet")
print("Notebook A output found at:", A_OUT)

WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
OUT  = WORK / 'darkline_bundle'; OUT.mkdir(exist_ok=True)

station_times = pd.read_parquet(A_OUT / 'station_times.parquet')
parts         = pd.read_parquet(A_OUT / 'parts.parquet')
catalog       = pd.read_parquet(A_OUT / 'station_catalog.parquet')
with open(A_OUT / 'column_map.json') as f:
    cmap = json.load(f)
stations = cmap['stations']

print("station_times", station_times.shape)
print("parts        ", parts.shape)
print("catalog      ", catalog.shape)

DARK = catalog.loc[catalog.is_dark, 'station'].tolist()
MEAS = catalog.loc[~catalog.is_dark, 'station'].tolist()
print(f"\nDARK stations ({len(DARK)}): {DARK}")

def dump(obj, name):
    with open(OUT / name, 'w') as f:
        json.dump(obj, f, default=lambda o: float(o) if isinstance(o, (np.floating,)) else int(o))
    print(f"[export] {name}  ({(OUT/name).stat().st_size/1024:.1f} KB)")

class Timer:
    def __init__(s, l): s.l = l
    def __enter__(s): s.t=time.time(); print(f"\n>>> {s.l}", flush=True); return s
    def __exit__(s,*a): print(f"<<< {s.l} — {time.time()-s.t:.1f}s", flush=True)

Notebook A output found at: /kaggle/input/notebooks/muskank23/darkline-a-extraction
station_times (1183747, 53)
parts         (1183747, 8)
catalog       (52, 8)

DARK stations (2): ['L3_S42', 'L3_S46']


In [20]:
# =============================================================
# ENGINE 1 — Per-station dwell, and WIP in each segment.
#
# Dwell at station s for a part = (time it reaches the NEXT station
# it visits) - (time it reached s). WIP in a segment at time t =
# how many parts are inside that segment at t, computed with a
# sorted event sweep: +1 on entry, -1 on exit, cumulative sum.
# O(n log n), not O(n^2).
# =============================================================

M = station_times[stations].to_numpy(dtype=np.float32)
ids = station_times['Id'].to_numpy()
visited = ~np.isnan(M)
order = np.argsort(np.where(visited, M, np.inf), axis=1)
nvis = visited.sum(axis=1)

with Timer("Computing per-station dwell"):
    dwell = np.full(M.shape, np.nan, dtype=np.float32)
    rows = np.arange(M.shape[0])
    for k in range(M.shape[1] - 1):
        cur, nxt = order[:, k], order[:, k+1]
        ok = (k + 1) < nvis
        d = M[rows[ok], nxt[ok]] - M[rows[ok], cur[ok]]
        dwell[rows[ok], cur[ok]] = d

dwell_df = pd.DataFrame(dwell, columns=[f'dwell_{s}' for s in stations])
dwell_df.insert(0, 'Id', ids)

print("Median dwell per station (Bosch time units):")
med = dwell_df[[c for c in dwell_df.columns if c!='Id']].median()
summary = pd.DataFrame({
    'station':[c.replace('dwell_','') for c in med.index],
    'median_dwell': med.values,
    'coverage': (dwell_df[[c for c in dwell_df.columns if c!='Id']].notna().mean().values*100).round(1),
}).merge(catalog[['station','instrumentation']], on='station')
print(summary.to_string(index=False))


def wip_series(entry, exit_, grid):
    """Number of parts inside a segment at each point of `grid`."""
    ok = ~(np.isnan(entry) | np.isnan(exit_))
    e, x = np.sort(entry[ok]), np.sort(exit_[ok])
    return np.searchsorted(e, grid, 'right') - np.searchsorted(x, grid, 'right')

print("\nWIP sweep function ready.")


>>> Computing per-station dwell
<<< Computing per-station dwell — 1.5s
Median dwell per station (Bosch time units):
station  median_dwell  coverage instrumentation
  L0_S0      0.000000      56.9        MEASURED
  L0_S1      0.000000      56.9        MEASURED
  L0_S2      0.020004      28.7        MEASURED
  L0_S3      0.020000      28.3        MEASURED
  L0_S4      0.000000      28.3        MEASURED
  L0_S5      0.000000      28.7        MEASURED
  L0_S6      0.000000      28.6        MEASURED
  L0_S7      0.000000      28.4        MEASURED
  L0_S8      0.000000      56.9        MEASURED
  L0_S9      1.899963      19.1        MEASURED
 L0_S10      1.900024      19.0        MEASURED
 L0_S11      1.900024      19.0        MEASURED
 L0_S12      0.000000      20.4        MEASURED
 L0_S13      0.010010      20.4        MEASURED
 L0_S14      0.010002      10.2        MEASURED
 L0_S15      0.009995      10.3        MEASURED
 L0_S16      0.000000      10.1        MEASURED
 L0_S17      0.0000

In [21]:

# =============================================================
# MEASUREMENT COVERAGE — the honest version of "dark"
# For every station visit (part passed through, we have a date),
# did any numeric feature actually get recorded?
# =============================================================
with open(A_OUT / 'column_map.json') as f:
    num_by_station = json.load(f)['num_by_station']

num_all = [c for c in pd.read_csv(NUM, nrows=0).columns if c not in ('Id', 'Response')]
has_meas = {s: np.zeros(len(station_times), dtype=bool) for s in stations}
id_pos = pd.Series(np.arange(len(station_times)), index=station_times['Id'].values)

with Timer("Scanning train_numeric for measurement presence"):
    CH = 50_000
    rdr = pd.read_csv(NUM, usecols=['Id'] + num_all,
                      dtype={c: np.float32 for c in num_all}, chunksize=CH)
    for i, chunk in enumerate(rdr):
        pos = id_pos.reindex(chunk['Id'].values).to_numpy()
        ok = ~np.isnan(pos)
        pos = pos[ok].astype(int)
        for s in stations:
            cols = num_by_station.get(s, [])
            if not cols:
                continue
            any_val = chunk.loc[ok, cols].notna().any(axis=1).to_numpy()
            has_meas[s][pos] |= any_val
        del chunk; gc.collect()
        if i % 5 == 0:
            print(f"   chunk {i}", flush=True)

visited_mat = ~np.isnan(station_times[stations].to_numpy(dtype=np.float32))
rows = []
for j, s in enumerate(stations):
    v = visited_mat[:, j]
    n_vis = int(v.sum())
    n_meas = int((v & has_meas[s]).sum())
    rows.append({'station': s, 'n_visits': n_vis, 'n_measured': n_meas,
                 'measured_pct': 100.0 * n_meas / max(n_vis, 1),
                 'n_numeric_cols': len(num_by_station.get(s, []))})

cov = pd.DataFrame(rows).sort_values('measured_pct')
print(cov.to_string(index=False))

total_visits = int(cov['n_visits'].sum())
total_meas   = int(cov['n_measured'].sum())
unmeasured_visit_pct = 100.0 * (1 - total_meas / total_visits)

STRUCT_DARK = cov.loc[cov.n_numeric_cols == 0, 'station'].tolist()
EFF_DARK    = cov.loc[cov.measured_pct < 5, 'station'].tolist()
LOW_COV     = cov.loc[cov.measured_pct < 50, 'station'].tolist()

print("\n" + "="*66)
print("THE HEADLINE NUMBERS")
print("="*66)
print(f"Stations with zero numeric columns (structurally dark) : {len(STRUCT_DARK)} / {len(stations)}")
print(f"Stations measuring <5% of the parts that pass through  : {len(EFF_DARK)} / {len(stations)}")
print(f"Stations measuring <50% of the parts passing through   : {len(LOW_COV)} / {len(stations)}")
print(f"SHARE OF ALL STATION VISITS WITH NO MEASUREMENT        : {unmeasured_visit_pct:.1f}%")
print("="*66)
print(">>> The last line is the headline. Send it to Kartik.")

cov.to_parquet(WORK / 'measurement_coverage.parquet', index=False)
dump({'per_station': cov.to_dict('records'),
      'n_structurally_dark': len(STRUCT_DARK),
      'n_effectively_dark_lt5pct': len(EFF_DARK),
      'n_low_coverage_lt50pct': len(LOW_COV),
      'n_stations': len(stations),
      'total_visits': total_visits,
      'total_measured_visits': total_meas,
      'unmeasured_visit_pct': unmeasured_visit_pct,
      'definition': ('Structurally dark = zero numeric feature columns. '
                     'Effectively dark = fewer than 5% of transiting parts receive '
                     'any numeric reading. The headline figure is the share of all '
                     'station visits across the line that produce no measurement.')},
     'measurement_coverage.json')


>>> Scanning train_numeric for measurement presence
   chunk 0
   chunk 5
   chunk 10
   chunk 15
   chunk 20
<<< Scanning train_numeric for measurement presence — 83.5s
station  n_visits  n_measured  measured_pct  n_numeric_cols
 L3_S42        15           0      0.000000               0
 L3_S46         1           0      0.000000               0
 L0_S22     80601       80599     99.997519              14
  L0_S9    225679      225678     99.999557              12
  L0_S4    335295      335295    100.000000               2
  L0_S5    339512      339512    100.000000               2
  L0_S2    339774      339774    100.000000               9
  L0_S0    673862      673862    100.000000              12
  L0_S7    335698      335698    100.000000               3
  L0_S8    673881      673881    100.000000               3
 L0_S10    224540      224540    100.000000              12
 L0_S11    225452      225452    100.000000              12
 L0_S12    242061      242061    100.000000      

In [22]:
# =============================================================
# ENGINE 2 — RECONSTRUCTION, AND ITS HONEST VALIDATION.
#
# The claim: for a station with no scan point at all, we can still
# recover its dwell time from the timestamps of its neighbours plus
# the traffic in that segment.
#
# The test: take stations we DO observe, HIDE them, reconstruct
# them, and score against the truth we hid from ourselves.
#
# HARD GATE: 90% prediction-interval coverage must land in
# [0.85, 0.95]. If it doesn't, the intervals are lying — fix the
# model, do NOT widen the claim.
# =============================================================
import lightgbm as lgb

SAMPLE = 250_000                     # keeps this cell to a few minutes
rng = np.random.default_rng(42)
sub = rng.choice(len(parts), size=min(SAMPLE, len(parts)), replace=False)
sub.sort()

train_mask = (parts['split'].to_numpy()[sub] == 'train')

def reconstruct_station(target_idx, verbose=True):
    """Hide station `target_idx`; predict its dwell from neighbours."""
    tgt = stations[target_idx]
    y_true = dwell[sub, target_idx]
    ok = ~np.isnan(y_true)
    if ok.sum() < 5000:
        return None

    # Position of the target in each part's route
    pos = np.argmax(order[sub] == target_idx, axis=1)

    feats = {}
    rows = np.arange(len(sub))
    # prev / next observed station indices around the hidden one
    prev_i = np.where(pos > 0, order[sub, np.maximum(pos-1, 0)], -1)
    next_i = np.where(pos+1 < nvis[sub], order[sub, np.minimum(pos+1, M.shape[1]-1)], -1)

    t_prev = np.where(prev_i >= 0, M[sub, prev_i], np.nan)
    t_here = M[sub, target_idx]
    t_next = np.where(next_i >= 0, M[sub, next_i], np.nan)

    # The interval we would actually observe if this station were unscanned
    feats['gap_prev_next'] = t_next - t_prev
    feats['route_len']     = nvis[sub]
    feats['position']      = pos
    feats['total_cycle']   = parts['total_cycle_time'].to_numpy()[sub]
    feats['entry_time']    = t_prev

    # Segment traffic: how busy the line was when this part went through
    grid = t_prev
    feats['wip_segment'] = wip_series(t_prev, t_next, np.nan_to_num(grid, nan=0.0)).astype(np.float32)

    # Neighbouring dwell times (observed — these stations keep their scans)
    for off, nm in [(-2,'dw_m2'), (1,'dw_p1'), (2,'dw_p2')]:
        p2 = pos + off
        valid = (p2 >= 0) & (p2 < nvis[sub])
        idx = order[sub, np.clip(p2, 0, M.shape[1]-1)]
        v = np.where(valid, dwell[sub, idx], np.nan)
        feats[nm] = v

    X = pd.DataFrame(feats).iloc[ok]
    y = y_true[ok]
    m_tr = train_mask[ok]

    if m_tr.sum() < 2000 or (~m_tr).sum() < 500:
        return None

    preds = {}
    for a, tag in [(0.05, 'lo'), (0.50, 'mid'), (0.95, 'hi')]:
        mdl = lgb.LGBMRegressor(objective='quantile', alpha=a,
                                n_estimators=250, learning_rate=0.06,
                                num_leaves=31, min_child_samples=40,
                                verbose=-1, random_state=42)
        mdl.fit(X[m_tr], y[m_tr])
        preds[tag] = mdl.predict(X[~m_tr])

    yt = y[~m_tr]
    err = np.abs(preds['mid'] - yt)
    cover = float(((yt >= preds['lo']) & (yt <= preds['hi'])).mean())
    res = {
        'station': tgt,
        'instrumentation': 'MEASURED' if tgt in MEAS else 'DARK',
        'n_test': int(len(yt)),
        'mae': float(np.mean(err)),
        'median_ae': float(np.median(err)),
        'coverage_90': cover,
        'baseline_mae': float(np.mean(np.abs(np.median(y[m_tr]) - yt))),
    }
    res['skill_vs_baseline'] = float(1 - res['mae'] / res['baseline_mae'])
    if verbose:
        print(f"  {tgt:10s} n={res['n_test']:7,d}  MAE={res['mae']:7.3f}  "
              f"median={res['median_ae']:7.3f}  coverage={cover:5.3f}  "
              f"skill={res['skill_vs_baseline']:+.3f}")
    return res, (yt, preds['mid'])


with Timer("Withhold-and-recover across stations"):
    order_by_cov = summary.sort_values('coverage', ascending=False)['station'].tolist()
    targets = [stations.index(s) for s in order_by_cov[:14]]
    results, scatter = [], None
    for ti in targets:
        r = reconstruct_station(ti)
        if r is None:
            continue
        res, sc = r
        results.append(res)
        if scatter is None and res['n_test'] > 3000:
            take = np.random.default_rng(0).choice(len(sc[0]), size=min(600, len(sc[0])), replace=False)
            scatter = {'station': res['station'],
                       'truth': sc[0][take].tolist(),
                       'pred':  sc[1][take].tolist()}

recon = pd.DataFrame(results)
mean_cov = float(recon['coverage_90'].mean())
print("\n" + "="*66)
print(f"MEAN 90% INTERVAL COVERAGE : {mean_cov:.3f}")
print(f"MEAN MAE                   : {recon['mae'].mean():.3f} time units")
print(f"MEAN SKILL vs baseline     : {recon['skill_vs_baseline'].mean():+.3f}")
if 0.85 <= mean_cov <= 0.95:
    print("STATUS: PASS — intervals are honest. Proceed.")
else:
    print("STATUS: FAIL — intervals are miscalibrated.")
    print("        Widen/narrow the quantile alphas or add features.")
    print("        DO NOT proceed and DO NOT weaken the claim on the slide.")
print("="*66)

dump({'summary': {'mean_coverage_90': mean_cov,
                  'mean_mae': float(recon['mae'].mean()),
                  'mean_skill': float(recon['skill_vs_baseline'].mean()),
                  'gate_passed': bool(0.85 <= mean_cov <= 0.95)},
      'per_station': recon.to_dict('records'),
      'scatter': scatter}, 'reconstruction_eval.json')


>>> Withhold-and-recover across stations
  L3_S29     n= 71,139  MAE=  0.005  median=  0.000  coverage=0.959  skill=+0.298
  L3_S30     n= 71,181  MAE=  0.007  median=  0.000  coverage=0.950  skill=+0.407
  L3_S34     n= 70,245  MAE=  0.003  median=  0.000  coverage=0.994  skill=+0.401
  L3_S33     n= 66,516  MAE=  0.008  median=  0.000  coverage=0.982  skill=+0.370
  L0_S0      n= 42,272  MAE=  0.001  median=  0.000  coverage=0.989  skill=+0.000
  L0_S1      n= 42,286  MAE=  0.002  median=  0.000  coverage=0.994  skill=+0.761
  L0_S8      n= 42,272  MAE=  0.005  median=  0.000  coverage=0.996  skill=+0.591
  L3_S37     n= 32,523  MAE=  0.037  median=  0.000  coverage=0.995  skill=+0.918
  L3_S35     n= 28,329  MAE=  0.018  median=  0.000  coverage=0.959  skill=+0.507
  L0_S2      n= 21,390  MAE=  0.012  median=  0.000  coverage=0.961  skill=+0.314
  L0_S5      n= 21,271  MAE=  0.003  median=  0.000  coverage=0.988  skill=+0.463
  L0_S6      n= 21,198  MAE=  0.002  median=  0.000  cov

In [23]:
# =============================================================
# ENGINE 3 — BOTTLENECK DETECTION.
#
# Active Period Method (Roser, Nakano & Tanaka, 2001): the station
# with the longest average uninterrupted active period is the
# momentary bottleneck. It needs only working-vs-waiting states —
# which is exactly what dwell times give us. That is why this
# method and not utilisation ratios.
# =============================================================

tmin, tmax = np.nanpercentile(parts['first_timestamp'], [1, 99])
N_WIN = 24
edges = np.linspace(tmin, tmax, N_WIN + 1)

with Timer("Bottleneck share per station per window"):
    windows = []
    for w in range(N_WIN):
        lo, hi = edges[w], edges[w+1]
        in_win = (M >= lo) & (M < hi)
        active = np.where(in_win & ~np.isnan(dwell), dwell, np.nan)
        with np.errstate(all='ignore'):
            mean_active = np.nanmean(active, axis=0)
            n_obs = (~np.isnan(active)).sum(axis=0)
        mean_active = np.where(n_obs > 200, mean_active, np.nan)
        if np.all(np.isnan(mean_active)):
            continue
        bidx = int(np.nanargmax(mean_active))
        tot = np.nansum(mean_active)
        windows.append({
            'window': w,
            't_start': float(lo), 't_end': float(hi),
            'bottleneck': stations[bidx],
            'is_dark': bool(stations[bidx] in DARK),
            'shares': {stations[i]: float(mean_active[i] / tot)
                       for i in range(len(stations))
                       if not np.isnan(mean_active[i])},
        })

migration = [(w['window'], w['bottleneck'], w['is_dark']) for w in windows]
print("Bottleneck by window:")
for w, b, d in migration:
    print(f"   window {w:2d}  ->  {b:10s} {'[DARK STATION]' if d else ''}")

from collections import Counter
cnt = Counter(b for _, b, _ in migration)
print("\nMost frequent constraint:", cnt.most_common(5))
dark_share = sum(1 for _, _, d in migration if d) / max(len(migration), 1)
print(f"Share of windows where the constraint sits in a DARK station: {dark_share*100:.1f}%")
print(">>> If this is above zero, it is direct evidence for our thesis.")

dump({'windows': windows,
      'migration': [{'window': w, 'station': b, 'is_dark': d} for w, b, d in migration],
      'dark_constraint_share': dark_share,
      'method': 'Active Period Method (Roser, Nakano & Tanaka 2001)'},
     'constraint.json')


>>> Bottleneck share per station per window


/tmp/ipykernel_58/4210094020.py:22: RuntimeWarning: Mean of empty slice
  mean_active = np.nanmean(active, axis=0)


<<< Bottleneck share per station per window — 20.4s
Bottleneck by window:
   window  0  ->  L2_S27     
   window  1  ->  L2_S27     
   window  2  ->  L2_S26     
   window  3  ->  L2_S26     
   window  4  ->  L1_S25     
   window  5  ->  L1_S25     
   window  6  ->  L1_S24     
   window  7  ->  L0_S9      
   window  8  ->  L1_S25     
   window  9  ->  L2_S27     
   window 10  ->  L1_S25     
   window 11  ->  L1_S25     
   window 12  ->  L2_S28     
   window 13  ->  L1_S25     
   window 14  ->  L1_S25     
   window 15  ->  L1_S25     
   window 16  ->  L1_S25     
   window 17  ->  L1_S25     
   window 18  ->  L1_S24     
   window 19  ->  L0_S10     
   window 20  ->  L1_S25     
   window 21  ->  L2_S26     
   window 22  ->  L2_S26     
   window 23  ->  L1_S24     

Most frequent constraint: [('L1_S25', 11), ('L2_S26', 4), ('L2_S27', 3), ('L1_S24', 3), ('L0_S9', 1)]
Share of windows where the constraint sits in a DARK station: 0.0%
>>> If this is above zero, it is dir

In [24]:
# =============================================================
# ENGINE 4 — DEFECT RISK, REBUILT WITHOUT LEAKAGE.
#
# Four controls, all of which the original notebook was missing:
#   1. temporal split, never random
#   2. feature screening on TRAIN ROWS ONLY
#   3. threshold chosen on VALIDATION, applied unchanged to test
#   4. isotonic calibration so the score is a real probability
# =============================================================
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (average_precision_score, matthews_corrcoef,
                             precision_recall_curve)

tr = (parts['split'] == 'train').to_numpy()
va = (parts['split'] == 'val').to_numpy()
te = (parts['split'] == 'test').to_numpy()
y  = parts['Response'].fillna(0).to_numpy().astype(np.int8)

# ---- screen numeric features on TRAIN ROWS ONLY -----------------
train_ids = set(parts.loc[tr, 'Id'].tolist())
all_num = [c for c in pd.read_csv(NUM, nrows=0).columns if c not in ('Id', 'Response')]

with Timer("Screening numeric features on TRAIN rows only"):
    scores = []
    B = 120
    for i in range(0, len(all_num), B):
        cols = all_num[i:i+B]
        d = pd.read_csv(NUM, usecols=['Id'] + cols,
                        dtype={c: np.float32 for c in cols})
        d = d[d['Id'].isin(train_ids)]
        yy = parts.set_index('Id').loc[d['Id'], 'Response'].to_numpy()
        for c in cols:
            v = d[c].to_numpy()
            m = ~np.isnan(v)
            if m.sum() < 3000: continue
            r = np.corrcoef(v[m], yy[m])[0, 1]
            if np.isfinite(r): scores.append((c, abs(r)))
        del d; gc.collect()
        if i % 600 == 0: print(f"   {i}/{len(all_num)}", flush=True)

    scores.sort(key=lambda t: -t[1])
    TOP = [c for c, _ in scores[:60]]
print("Top screened features:", TOP[:10])

with Timer("Assembling the feature matrix"):
    numeric_top = pd.read_csv(NUM, usecols=['Id'] + TOP,
                              dtype={c: np.float32 for c in TOP})
    dwell_keep = [f'dwell_{s}' for s in stations]
    X = (parts[['Id', 'total_cycle_time', 'n_stations_visited']]
         .merge(dwell_df[['Id'] + dwell_keep], on='Id', how='left')
         .merge(numeric_top, on='Id', how='left'))
    path_freq = parts['path_signature'].map(
        parts.loc[tr, 'path_signature'].value_counts(normalize=True)).fillna(0)
    X['path_frequency'] = path_freq.to_numpy()
    X = X.drop(columns=['Id'])
    for c in X.columns:
        X[c] = X[c].astype(np.float32)
    del numeric_top; gc.collect()
print("Feature matrix:", X.shape)

spw = (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)
base = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=63,
                          max_depth=8, min_child_samples=40,
                          scale_pos_weight=spw, verbose=-1, random_state=42)
with Timer("Fitting on TRAIN only"):
    base.fit(X[tr], y[tr])

with Timer("Isotonic calibration on VALIDATION"):
    cal = CalibratedClassifierCV(base, method='isotonic', cv='prefit')
    cal.fit(X[va], y[va])

p_va, p_te = cal.predict_proba(X[va])[:, 1], cal.predict_proba(X[te])[:, 1]

# threshold chosen on validation ONLY
ths = np.quantile(p_va, np.linspace(0.90, 0.9995, 120))
mccs = [matthews_corrcoef(y[va], (p_va > t).astype(int)) for t in ths]
THRESH = float(ths[int(np.argmax(mccs))])
print(f"\nThreshold chosen on validation: {THRESH:.5f}  (val MCC {max(mccs):.4f})")

def boot(yv, pv, fn, n=200):
    rs = np.random.default_rng(7); out = []
    for _ in range(n):
        i = rs.integers(0, len(yv), len(yv))
        try: out.append(fn(yv[i], pv[i]))
        except Exception: pass
    return [float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))]

pred_te = (p_te > THRESH).astype(int)
report = {
    'split_sizes': {'train': int(tr.sum()), 'val': int(va.sum()), 'test': int(te.sum())},
    'test_failure_rate': float(y[te].mean()),
    'pr_auc':  float(average_precision_score(y[te], p_te)),
    'pr_auc_ci': boot(y[te], p_te, average_precision_score),
    'mcc':     float(matthews_corrcoef(y[te], pred_te)),
    'threshold': THRESH,
    'n_alerts': int(pred_te.sum()),
    'precision': float(y[te][pred_te == 1].mean()) if pred_te.sum() else 0.0,
    'recall':    float(pred_te[y[te] == 1].mean()) if (y[te] == 1).sum() else 0.0,
    'lift_over_random': float(average_precision_score(y[te], p_te) / max(y[te].mean(), 1e-9)),
    'leakage_controls': [
        'Temporal split by first_timestamp (70/15/15), never random',
        'Numeric feature screening computed on TRAIN rows only',
        'Operating threshold selected on VALIDATION, applied unchanged to TEST',
        'Isotonic calibration fitted on VALIDATION so scores are probabilities',
        'TEST split scored exactly once',
    ],
}
for k in [50, 100, 500, 1000]:
    top = np.argsort(-p_te)[:k]
    report[f'precision_at_{k}'] = float(y[te][top].mean())

pr, rc, _ = precision_recall_curve(y[te], p_te)
step = max(len(pr)//400, 1)
report['pr_curve'] = {'precision': pr[::step].tolist(), 'recall': rc[::step].tolist()}

# calibration reliability
bins = np.quantile(p_te, np.linspace(0, 1, 11))
report['calibration'] = []
for i in range(10):
    m = (p_te >= bins[i]) & (p_te < bins[i+1])
    if m.sum() > 50:
        report['calibration'].append({'predicted': float(p_te[m].mean()),
                                      'observed': float(y[te][m].mean()),
                                      'n': int(m.sum())})

# cost curve — what a plant manager actually decides on
C_INSPECT, C_ESCAPE = 1.0, 60.0
report['cost_curve'] = []
for q in np.linspace(0.90, 0.9995, 40):
    t = float(np.quantile(p_va, q))
    pd_ = (p_te > t).astype(int)
    fn_ = int(((pd_ == 0) & (y[te] == 1)).sum())
    cost = C_INSPECT * int(pd_.sum()) + C_ESCAPE * fn_
    report['cost_curve'].append({'threshold': t, 'n_alerts': int(pd_.sum()),
                                 'escapes': fn_, 'cost': float(cost)})
best = min(report['cost_curve'], key=lambda r: r['cost'])
report['cost_optimal'] = best

print("\n" + "="*66)
print("HELD-OUT TEST RESULTS  (this is what goes on the slide)")
print("="*66)
for k in ['pr_auc','mcc','precision','recall','lift_over_random',
          'precision_at_100','n_alerts','test_failure_rate']:
    print(f"  {k:22s} {report[k]:.4f}")
print(f"  cost-optimal threshold {best['threshold']:.4f} -> "
      f"{best['n_alerts']} alerts, {best['escapes']} escapes")
print("="*66)

dump(report, 'model_report.json')


>>> Screening numeric features on TRAIN rows only
   0/968


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


   600/968
<<< Screening numeric features on TRAIN rows only — 316.2s
Top screened features: ['L1_S24_F867', 'L1_S24_F1723', 'L1_S24_F839', 'L1_S24_F1632', 'L1_S24_F1695', 'L1_S24_F1604', 'L1_S24_F1758', 'L1_S24_F902', 'L1_S24_F1667', 'L1_S24_F1846']

>>> Assembling the feature matrix
<<< Assembling the feature matrix — 34.7s
Feature matrix: (1183747, 115)

>>> Fitting on TRAIN only
<<< Fitting on TRAIN only — 31.2s

>>> Isotonic calibration on VALIDATION


/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


<<< Isotonic calibration on VALIDATION — 5.3s

Threshold chosen on validation: 0.07692  (val MCC 0.1622)

HELD-OUT TEST RESULTS  (this is what goes on the slide)
  pr_auc                 0.0094
  mcc                    0.0659
  precision              0.0700
  recall                 0.0688
  lift_over_random       2.4924
  precision_at_100       0.0600
  n_alerts               657.0000
  test_failure_rate      0.0038
  cost-optimal threshold 0.0150 -> 840 alerts, 618 escapes
[export] model_report.json  (5.6 KB)


In [25]:
# =============================================================
# ENGINE 5 — THE ABLATION.
#
# Our whole pitch says under-measured stations matter. Here is a real
# test of that: train the same model twice, once WITHOUT any
# low-coverage station timing feature, once WITH. If PR-AUC improves,
# then stations that measure little or nothing carry recoverable
# signal — proven on real data, not asserted on a slide.
# =============================================================

dark_cols = [f'dwell_{s}' for s in LOW_COV if f'dwell_{s}' in X.columns]
print(f"Low-coverage dwell features in the matrix: {len(dark_cols)}")
assert len(dark_cols) > 0, (
    "Ablation is meaningless with zero columns to remove. "
    "Check that dwell_{station} columns exist for the LOW_COV list.")

def fit_score(cols, label):
    Xs = X[cols]
    m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=63,
                           max_depth=8, min_child_samples=40,
                           scale_pos_weight=spw, verbose=-1, random_state=42)
    m.fit(Xs[tr], y[tr])
    p = m.predict_proba(Xs[te])[:, 1]
    ap = float(average_precision_score(y[te], p))
    print(f"   {label:34s} PR-AUC = {ap:.5f}")
    return ap

with Timer("Ablation: with and without low-coverage stations"):
    without = fit_score([c for c in X.columns if c not in dark_cols], "WITHOUT low-coverage timing")
    withd   = fit_score(list(X.columns),                              "WITH low-coverage timing")

delta = withd - without
rel = delta / max(without, 1e-9) * 100
print("\n" + "="*66)
print(f"Delta PR-AUC from low-coverage stations: {delta:+.5f}  ({rel:+.1f}% relative)")
print("VERDICT:", "low-coverage stations carry recoverable signal"
      if delta > 0 else "no measurable gain — report this honestly")
print("="*66)

dump({'pr_auc_without_lowcov': without, 'pr_auc_with_lowcov': withd,
      'delta': delta, 'relative_pct': rel,
      'n_lowcov_features': len(dark_cols),
      'verdict': 'low-coverage stations carry signal' if delta > 0 else 'no measurable gain'},
     'ablation.json')

Low-coverage dwell features in the matrix: 2

>>> Ablation: with and without low-coverage stations
   WITHOUT low-coverage timing        PR-AUC = 0.01101
   WITH low-coverage timing           PR-AUC = 0.01101
<<< Ablation: with and without low-coverage stations — 61.3s

Delta PR-AUC from low-coverage stations: +0.00000  (+0.0% relative)
VERDICT: no measurable gain — report this honestly
[export] ablation.json  (0.2 KB)


In [26]:
# =============================================================
# ENGINE 6 — DRIFT, CHANGE-POINT AND BLAST RADIUS.
#
# Control limits are +/-3 sigma of the feature's own early stable
# regime. They are STATISTICAL control limits derived from data.
# They are NOT engineering spec limits — we do not know Bosch's
# real specs and we never claim to.
# =============================================================
try:
    import ruptures as rpt
    HAVE_RPT = True
except ImportError:
    !pip install -q ruptures
    import ruptures as rpt
    HAVE_RPT = True

CAND = [c for c, _ in scores[:25]]
best_drift, drift_payload = None, None

for col in CAND:
    d = pd.read_csv(NUM, usecols=['Id', col], dtype={col: np.float32})
    d = d.merge(parts[['Id', 'first_timestamp', 'split', 'Response']], on='Id')
    d = d.dropna(subset=[col]).sort_values('first_timestamp')
    if len(d) < 20_000:
        continue

    K = 400
    d['bin'] = pd.qcut(d['first_timestamp'], K, labels=False, duplicates='drop')
    g = d.groupby('bin').agg(t=('first_timestamp', 'mean'),
                             v=(col, 'median'),
                             n=(col, 'size')).dropna()
    if len(g) < 100: continue

    stable = g['v'].iloc[:len(g)//4]
    mu, sd = stable.mean(), stable.std()
    if not np.isfinite(sd) or sd == 0: continue
    z = (g['v'] - mu) / sd

    algo = rpt.Pelt(model='rbf', min_size=15).fit(g['v'].to_numpy())
    bkps = algo.predict(pen=6)
    if len(bkps) < 2: continue
    cp = bkps[0]
    post = z.iloc[cp:]
    drift_mag = abs(post.mean() - z.iloc[:cp].mean())
    if best_drift is None or drift_mag > best_drift[1]:
        best_drift = (col, float(drift_mag))
        drift_payload = {'feature': col, 'cp_index': int(cp),
                         'cp_time': float(g['t'].iloc[cp]),
                         't': g['t'].tolist(), 'v': g['v'].tolist(),
                         'z': z.tolist(), 'mu': float(mu), 'sd': float(sd),
                         'ucl': float(mu + 3*sd), 'lcl': float(mu - 3*sd),
                         'drift_magnitude_sigma': float(drift_mag)}
    del d; gc.collect()

if drift_payload is None:
    print("No clean drift event found in the screened features.")
    print(">>> Report this honestly. Do NOT manufacture one.")
    dump({'found': False,
          'note': 'No statistically clean drift event was found in the top screened '
                  'features on the real Bosch data. Not fabricated.'}, 'drift.json')
else:
    f, cp_t = drift_payload['feature'], drift_payload['cp_time']
    print(f"Strongest drift: {f}  change-point at t={cp_t:.1f}  "
          f"magnitude {drift_payload['drift_magnitude_sigma']:.2f} sigma")

    # ---- INDEPENDENT CORROBORATION (hard requirement) ----------
    # The boundary may only move if a PHYSICALLY SEPARATE signal
    # agrees that things were stable before the change-point.
    corrob = []
    for other in [c for c in CAND if c.split('_S')[1].split('_')[0] != f.split('_S')[1].split('_')[0]][:3]:
        d2 = pd.read_csv(NUM, usecols=['Id', other], dtype={other: np.float32})
        d2 = d2.merge(parts[['Id','first_timestamp']], on='Id').dropna().sort_values('first_timestamp')
        pre  = d2.loc[d2.first_timestamp < cp_t, other]
        post = d2.loc[d2.first_timestamp >= cp_t, other]
        if len(pre) < 1000 or len(post) < 1000: continue
        shift = abs(post.mean() - pre.mean()) / (pre.std() + 1e-9)
        corrob.append({'signal': other, 'station': 'S'+other.split('_S')[1].split('_')[0],
                       'shift_sigma': float(shift), 'stable': bool(shift < 0.5)})
        del d2; gc.collect()

    fr_pre  = parts.loc[parts.first_timestamp <  cp_t, 'Response'].mean()
    fr_post = parts.loc[parts.first_timestamp >= cp_t, 'Response'].mean()
    corrob.append({'signal': 'rolling_failure_rate', 'station': 'line',
                   'shift_sigma': float(abs(fr_post-fr_pre)/(fr_pre+1e-9)),
                   'stable': bool(abs(fr_post-fr_pre)/(fr_pre+1e-9) < 0.5)})

    n_ind = sum(1 for c in corrob if c['stable'])
    boundary_may_move = n_ind >= 1          # <-- ENFORCED, not a comment
    drift_payload['found'] = True
    drift_payload['corroboration'] = corrob
    drift_payload['independent_stable_signals'] = n_ind
    drift_payload['boundary_may_move'] = boundary_may_move

    # ---- CONTAINMENT -------------------------------------------
        # ---- CONTAINMENT -------------------------------------------
    dcol = pd.read_csv(NUM, usecols=['Id', f], dtype={f: np.float32})
    dcol = dcol.merge(parts[['Id','first_timestamp','Response']], on='Id').dropna()

    window_end = float(g['t'].iloc[-1])
    anchor     = float(g['t'].iloc[0])          # start of the observed series
    assert anchor < cp_t < window_end, f"bad window: {anchor} {cp_t} {window_end}"

    naive = dcol[(dcol.first_timestamp >= anchor) & (dcol.first_timestamp <= window_end)]

    if boundary_may_move:
        tgt = dcol[(dcol.first_timestamp >= cp_t) & (dcol.first_timestamp <= window_end)]
        tgt = tgt[(tgt[f] > drift_payload['ucl']) | (tgt[f] < drift_payload['lcl'])]
    else:
        tgt = naive
        print("Boundary NOT moved — no independent signal corroborates it.")

    assert len(tgt) <= len(naive), "containment set cannot exceed the naive set"
    reduction = 100.0 * (1 - len(tgt) / max(len(naive), 1))

    drift_payload['containment'] = {
        'naive_population': int(len(naive)),
        'darkline_population': int(len(tgt)),
        'reduction_pct': float(reduction),
        'sample_part_ids': tgt['Id'].head(40).tolist(),
    }
    print(f"Containment: {len(naive):,} -> {len(tgt):,} parts "
          f"({reduction:.1f}% reduction)")
    dump(drift_payload, 'drift.json')

Strongest drift: L1_S24_F1581  change-point at t=405.2  magnitude 1.65 sigma
Containment: 66,495 -> 20,016 parts (69.9% reduction)
[export] drift.json  (24.8 KB)


In [27]:
# =============================================================
# EXPORT — everything the dashboard reads.
#
# The browser cannot load 1.18M rows, so parts_sample.csv keeps
# every TEST-split alert plus a random sample of the rest.
# =============================================================

p_all = np.full(len(parts), np.nan)
p_all[te] = p_te
p_all[va] = p_va
out = parts[['Id','first_timestamp','last_timestamp','total_cycle_time',
             'n_stations_visited','path_signature','split','Response']].copy()
out['risk_score'] = p_all
out['alert'] = (out['risk_score'] > THRESH).fillna(False)

alerts = out[out.alert & (out.split == 'test')]
rest   = out[~out.index.isin(alerts.index)].sample(
             n=min(30_000, len(out)), random_state=42)
sample = pd.concat([alerts, rest]).drop_duplicates('Id')
sample.to_csv(OUT / 'parts_sample.csv', index=False)
print(f"[export] parts_sample.csv  {len(sample):,} rows "
      f"({len(alerts):,} test-split alerts + sample)")

# line overview
lo = []
for _, r in catalog.iterrows():
    s = r['station']
    dv = dwell_df[f'dwell_{s}'].dropna()
    rr = recon[recon.station == s]
    lo.append({
        'station': s, 'line': int(r['line']), 'station_no': int(r['station_no']),
        'instrumentation': r['instrumentation'], 'is_dark': bool(r['is_dark']),
        'n_numeric': int(r['n_numeric']), 'n_date': int(r['n_date']),
        'median_dwell': float(dv.median()) if len(dv) else None,
        'p25_dwell': float(dv.quantile(.25)) if len(dv) else None,
        'p75_dwell': float(dv.quantile(.75)) if len(dv) else None,
        'coverage_pct': float(dwell_df[f'dwell_{s}'].notna().mean()*100),
        'reconstruction_mae': float(rr['mae'].iloc[0]) if len(rr) else None,
        'reconstruction_coverage': float(rr['coverage_90'].iloc[0]) if len(rr) else None,
    })

dump({'stations': lo,
      'n_stations': len(catalog), 'n_dark': int(catalog.is_dark.sum()),
      'dark_pct': float(catalog.is_dark.mean()*100),
      'n_parts': int(len(parts)),
      'failure_rate': float(parts['Response'].mean()),
      'time_units_note': 'Bosch timestamps are anonymised relative units, not wall-clock.',
      'source': 'REAL'}, 'line_overview.json')

paths = (parts.groupby('path_signature')
         .agg(n=('Response','size'), fail_rate=('Response','mean'),
              median_cycle=('total_cycle_time','median'))
         .query('n > 3000').sort_values('n', ascending=False).head(25).reset_index())
dump({'paths': paths.to_dict('records'),
      'note': 'Failure rate varies by routing path — any pass/fail dwell '
              'comparison must control for path or it is confounded.'},
     'paths.json')

dump({'team':'CORTEX','institute':'IIT Kanpur',
      'dataset':'Bosch Production Line Performance',
      'generated_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
      'files':[f.name for f in sorted(OUT.glob('*'))]}, 'manifest.json')

zp = WORK / 'darkline_bundle.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.glob('*')):
        z.write(f, f.name)

print("\n" + "="*66)
print("BUNDLE READY —", zp)
print("="*66)
for f in sorted(OUT.glob('*')):
    print(f"   {f.name:32s} {f.stat().st_size/1024:8.1f} KB")
print("="*66)
print("Save Version -> download darkline_bundle.zip -> send to Kartik.")

[export] parts_sample.csv  30,657 rows (657 test-split alerts + sample)
[export] line_overview.json  (15.9 KB)
[export] paths.json  (3.4 KB)
[export] manifest.json  (0.3 KB)

BUNDLE READY — /kaggle/working/darkline_bundle.zip
   ablation.json                         0.2 KB
   constraint.json                      35.2 KB
   drift.json                           24.8 KB
   line_overview.json                   15.9 KB
   manifest.json                         0.3 KB
   measurement_coverage.json             6.0 KB
   model_report.json                     5.6 KB
   parts_sample.csv                   2639.3 KB
   paths.json                            3.4 KB
   reconstruction_eval.json             23.5 KB
Save Version -> download darkline_bundle.zip -> send to Kartik.


In [1]:
from IPython.display import FileLink
FileLink('darkline_bundle.zip')

/kaggle/working/darkline_bundle.zip